In [22]:
import datetime
import pandas as pdiddy

date = datetime.datetime.fromtimestamp(27920853362000 / 1e3)

from_no_to_time = lambda x: datetime.datetime.fromtimestamp(x / 1e3)

df = pdiddy.read_csv('../ignore_files/data/full/Phones_accelerometer.csv')

df['Creation_Time'] = pdiddy.to_datetime(df['Creation_Time'], unit='ns')

print(df.head())


   Index   Arrival_Time                 Creation_Time         x         y  \
0      0  1424696633908 2015-02-23 13:03:51.913248572 -5.958191  0.688065   
1      1  1424696633909 2015-02-23 13:03:51.918283972 -5.952240  0.670212   
2      2  1424696633918 2015-02-23 13:03:51.923288855 -5.995087  0.653549   
3      3  1424696633919 2015-02-23 13:03:51.928385290 -5.942718  0.676163   
4      4  1424696633929 2015-02-23 13:03:51.933420691 -5.991516  0.641647   

          z User   Model    Device     gt  
0  8.135345    a  nexus4  nexus4_1  stand  
1  8.136536    a  nexus4  nexus4_1  stand  
2  8.204376    a  nexus4  nexus4_1  stand  
3  8.128204    a  nexus4  nexus4_1  stand  
4  8.135345    a  nexus4  nexus4_1  stand  


In [ ]:

rows_per_second = (
    df.groupby(['Model', 'Device'])
    .agg(
        Observations=('Arrival_Time', 'size'),
        First_Observation=('Arrival_Time', 'min'),
        Last_Observation=('Arrival_Time', 'max'),
    )
    .assign(
        Duration_Seconds=lambda group: (
            group['Last_Observation'] - group['First_Observation']
        ).dt.total_seconds(),
    )
    .assign(
        Average_Rows_Per_Second=lambda group: (
            group['Observations'] / group['Duration_Seconds']
        ).where(group['Duration_Seconds'] > 0)
    )
    .reset_index()
    .sort_values(['Model', 'Device'])
)

rows_per_second.head()

rows_per_second.to_csv('rows_per_second.csv', index=False)